### IMPORTS

In [ ]:
import ollama
import tensorflow as tf
# import numpy as np
# import matplotlib.pyplot as plt
from tensorflow import keras
# from keras import Sequential, Input, layers, optimizers
# from keras.callbacks import EarlyStopping, ModelCheckpoint
# from keras.applications.vgg16 import VGG16
# from keras.applications.resnet50 import ResNet50
from keras.applications.efficientnet import EfficientNetB0
# from sklearn.metrics import classification_report, confusion_matrix
# import pandas as pd
# import seaborn as sns
# from PIL import Image

### CONFIGURATION

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
RANDOM_SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

### PATHS

In [ ]:
ORIGINAL_DIR = "../raw_data/Original"
AUGMENTED_DIR = "../raw_data/Augmented"

### LOAD DATASETS FROM DIRECTORY

In [ ]:
## Train dataset - Augmented data

train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    AUGMENTED_DIR,
    labels="inferred",
    label_mode="binary",
    class_names=["Non-Stone", "Stone"],
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=RANDOM_SEED
)

In [ ]:
## Full Original Dataset

full_original = tf.keras.preprocessing.image_dataset_from_directory(
    ORIGINAL_DIR,
    labels="inferred",
    label_mode="binary",
    class_names=["Non-Stone", "Stone"],
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=RANDOM_SEED
)

In [ ]:
full_original_shuffled = full_original.shuffle(
    buffer_size=len(full_original) * BATCH_SIZE,
    seed=RANDOM_SEED,
    reshuffle_each_iteration=False
)

### SPLIT ORIGINAL DATASET

In [ ]:
## Val and Test dataset - From Full Original dataset

total_batches = len(full_original)
val_batches = total_batches // 2

val_dataset = full_original_shuffled.take(val_batches).cache().prefetch(AUTOTUNE)
test_dataset = full_original_shuffled.skip(val_batches).cache().prefetch(AUTOTUNE)

### GET DATASETS

In [ ]:
def prepare(dataset, preprocess_input):
    return (
        dataset.map(lambda img, label: (preprocess_input(img), label), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
    )

In [ ]:
def get_datasets(preprocess_input):
    return (
        prepare(train_dataset, preprocess_input),
        prepare(val_dataset, preprocess_input),
        prepare(test_dataset, preprocess_input)
    )

In [ ]:
efficientnet_preprocess = tf.keras.applications.efficientnet.preprocess_input

In [ ]:
train_efficientnet, val_efficientnet, test_efficientnet = get_datasets(efficientnet_preprocess)

### LOAD MODEL

In [ ]:
efficientnet_model = keras.models.load_model("../models/efficientnet_best.keras")
print("Model loaded successfully!")

In [ ]:
efficientnet_model.summary()

### PREDICT

In [ ]:
def predict_with_confidence(model, img_tensor):
    prob = model.predict(img_tensor, verbose=0)  # raw sigmoid output e.g. 0.923
    prob = float(prob.flatten()[0])              # scalar

    if prob >= 0.5:
        label = "Stone"
        confidence = prob                        # e.g. 0.923 → 92.3% confident it's a Stone
    else:
        label = "Non-Stone"
        confidence = 1 - prob                   # e.g. 0.12 → 88% confident it's Non-Stone

    return label, round(confidence * 100, 2)    # returns ("Stone", 92.3)

### IMAGE SELECTION

In [ ]:
BATCH_NUM = 3
IDX = 5

for images, labels in test_efficientnet.skip(BATCH_NUM).take(1):
    # Take first image from the batch
    img_tensor = images[IDX:IDX+1]                        # shape (1, 224, 224, 3)
    true_label = int(labels[IDX].numpy())

sample_label, sample_confidence = predict_with_confidence(efficientnet_model, img_tensor)

img_pil = Image.fromarray(
    (images[IDX].numpy()).astype("uint8")
)

label_names = {0: "Non-Stone", 1: "Stone"}
print(f"True label : {label_names[true_label]}")
print(f"Prediction : {sample_label} ({sample_confidence}%)")
print(f"\\n--- CLINICAL REPORT ---\\n")

### GENERATE REPORT

In [ ]:
def generate_ollama_report(img_pil, prediction_label, confidence_score):
    import io, base64

    # Convert PIL image to base64
    buffer = io.BytesIO()
    img_pil.save(buffer, format="JPEG")
    img_base64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

    prompt = f"""You are a clinical radiology assistant specialized in kidney stone detection.

    An AI model has analyzed this CT scan and detected the following:
    - Detection result : {prediction_label}
    - Model confidence : {confidence_score}%

    Based on the image and the AI result above, generate a concise professional clinical report.
    Structure it as:

    FINDINGS:
    IMPRESSION:
    RECOMMENDATION:

    Rules:
    - Do NOT invent details not provided.
    - Do NOT contradict the AI detection result
    - Keep it under 200 words
    - Use formal medical language
    """

    response = ollama.chat(
        model="llava",
        messages=[{
            "role": "user",
            "content": prompt,
            "images": [img_base64]
        }]
    )

    return response["message"]["content"]

### TEST

In [ ]:
report = generate_ollama_report(img_pil, sample_label, sample_confidence)
print(report)